# ONH Binarization Experiments

This notebook is only for ONH binarization/preprocessing experiments. It does not change the GUI, Retina code, or the production ONH pipeline.

Current assumptions:
- All ONH images are treated as 4.5 mm x 4.5 mm scans.
- The analysis region is a fixed 2-4 mm diameter annulus.
- ONH center positions come from `test images information.xlsx`.
- The goal of this notebook is visual tuning of ONH binary masks on four representative images.

Important design choice in this version:
- Every candidate version explicitly preserves large bright vessels with a `large_seed` mask.
- The three versions mainly compare fine-vessel/noise behavior, not whether large vessels disappear.

In [ ]:
from pathlib import Path
import sys

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from scipy.ndimage import median_filter
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 200
plt.rcParams["font.size"] = 10


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "gui" / "build" / "modules" / "frangi_2Dfilter.py").exists():
            if (candidate / "algorithms test images").exists():
                return candidate
    raise FileNotFoundError("Could not find the OCTAVA project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
TEST_DIR = PROJECT_ROOT / "algorithms test images"
MODULE_DIR = PROJECT_ROOT / "gui" / "build" / "modules"

if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from frangi_2Dfilter import frangi_2Dfilter

METADATA_PATH = TEST_DIR / "test images information.xlsx"
OUTPUT_DIR = TEST_DIR / "notebook_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Test image folder:", TEST_DIR)
print("Metadata:", METADATA_PATH)

## Load The Four Test Images

The Excel file should contain: `filename`, `device`, `quality`, `center_x`, `center_y`.
The center coordinates are interpreted in the resized 1500 x 1500 scan coordinate system.

In [ ]:
IMAGE_SIZE_PX = 1500
SCAN_SIZE_MM = 4.5
INNER_DIAMETER_MM = 2.0
OUTER_DIAMETER_MM = 4.0

metadata = pd.read_excel(METADATA_PATH)
metadata

In [ ]:
def read_grayscale_image(path: Path) -> np.ndarray:
    image = Image.open(path).convert("L")
    return np.asarray(image, dtype=np.float64)


def resize_to_scan(image: np.ndarray, image_size_px: int = IMAGE_SIZE_PX) -> np.ndarray:
    interpolation = cv2.INTER_AREA if max(image.shape) > image_size_px else cv2.INTER_CUBIC
    resized = cv2.resize(image, (image_size_px, image_size_px), interpolation=interpolation)
    return resized.astype(np.float64)


def annulus_mask(
    image_shape: tuple[int, int],
    center_x: float,
    center_y: float,
    inner_diameter_mm: float = INNER_DIAMETER_MM,
    outer_diameter_mm: float = OUTER_DIAMETER_MM,
    scan_size_mm: float = SCAN_SIZE_MM,
) -> np.ndarray:
    h, w = image_shape
    px_per_mm = w / scan_size_mm
    inner_radius_px = (inner_diameter_mm / 2.0) * px_per_mm
    outer_radius_px = (outer_diameter_mm / 2.0) * px_per_mm
    yy, xx = np.ogrid[:h, :w]
    distance = np.sqrt((xx - center_x) ** 2 + (yy - center_y) ** 2)
    return (distance >= inner_radius_px) & (distance <= outer_radius_px)


def load_cases(metadata: pd.DataFrame) -> list[dict]:
    cases = []
    for _, row in metadata.iterrows():
        image_path = TEST_DIR / str(row["filename"])
        raw = read_grayscale_image(image_path)
        image = resize_to_scan(raw)
        center_x = float(row["center_x"])
        center_y = float(row["center_y"])
        roi_mask = annulus_mask(image.shape, center_x, center_y)
        device = str(row["device"]).strip()
        quality = str(row["quality"]).strip()
        cases.append(
            {
                "name": f"{device} {quality}",
                "filename": str(row["filename"]),
                "path": image_path,
                "raw_shape": raw.shape,
                "image": image,
                "roi_mask": roi_mask,
                "center_x": center_x,
                "center_y": center_y,
                "device": device,
                "quality": quality,
            }
        )
    return cases


cases = load_cases(metadata)
[(case["name"], case["raw_shape"], case["image"].shape, (case["center_x"], case["center_y"])) for case in cases]

In [ ]:
def normalize_for_display(image: np.ndarray, roi_mask: np.ndarray | None = None) -> np.ndarray:
    image = image.astype(np.float64)
    pixels = image.ravel() if roi_mask is None else image[roi_mask]
    if pixels.size == 0:
        return np.zeros_like(image, dtype=np.uint8)
    lo, hi = np.percentile(pixels, (1, 99))
    if hi <= lo:
        lo, hi = float(pixels.min()), float(pixels.max())
    out = (image - lo) / (hi - lo + 1e-8)
    out = np.clip(out, 0, 1)
    return (out * 255).astype(np.uint8)


def show_input_rois(cases: list[dict]) -> None:
    fig, axes = plt.subplots(1, len(cases), figsize=(4.2 * len(cases), 4.2))
    if len(cases) == 1:
        axes = [axes]
    for ax, case in zip(axes, cases):
        image = normalize_for_display(case["image"], case["roi_mask"])
        masked = np.where(case["roi_mask"], image, 0)
        ax.imshow(masked, cmap="gray", vmin=0, vmax=255)
        ax.scatter([case["center_x"]], [case["center_y"]], s=18, c="cyan")
        ax.set_title(case["name"])
        ax.axis("off")
    fig.suptitle("Input ROI: fixed 2-4 mm diameter annulus", y=0.98)
    plt.tight_layout()


show_input_rois(cases)

## Binarization Function

The core idea is split into two parts:

1. `large_seed`: preserves bright/large vessels so they do not disappear.
2. `fine_candidate`: uses Frangi vesselness to recover thin capillary-like vessels.

The final binary is `large_seed OR fine_candidate`, followed by a small cleanup close.

In [ ]:
def odd_kernel_size(image_size: int, scale: float, minimum: int) -> int:
    size = max(minimum, int(round(image_size * scale)))
    if size % 2 == 0:
        size += 1
    return size


def normalize_roi_to_uint8(image: np.ndarray, roi_mask: np.ndarray, low_pct: float = 1, high_pct: float = 99) -> np.ndarray:
    pixels = image[roi_mask]
    if pixels.size == 0:
        return np.zeros_like(image, dtype=np.uint8)
    lo, hi = np.percentile(pixels, (low_pct, high_pct))
    if hi <= lo:
        lo, hi = float(pixels.min()), float(pixels.max())
    out = (image - lo) / (hi - lo + 1e-8)
    out = np.clip(out, 0, 1)
    out = (out * 255).astype(np.uint8)
    return np.where(roi_mask, out, 0).astype(np.uint8)


def remove_small_components(binary: np.ndarray, roi_mask: np.ndarray, min_area_px: int) -> np.ndarray:
    if min_area_px <= 0:
        return np.where(roi_mask, binary, 0).astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    cleaned = np.zeros_like(binary, dtype=np.uint8)
    for label in range(1, num_labels):
        if stats[label, cv2.CC_STAT_AREA] >= min_area_px:
            cleaned[labels == label] = 255
    return np.where(roi_mask, cleaned, 0).astype(np.uint8)


def build_large_seed(enhanced: np.ndarray, corrected: np.ndarray, roi_mask: np.ndarray, image_size: int, params: dict) -> np.ndarray:
    source_name = params.get("large_seed_source", "enhanced")
    source = corrected if source_name == "corrected" else enhanced
    roi_pixels = source[roi_mask]

    candidate_pct = float(params.get("large_seed_percentile", 86))
    candidate_threshold = np.percentile(roi_pixels, candidate_pct)
    candidate = np.zeros_like(source, dtype=np.uint8)
    candidate[(source >= candidate_threshold) & roi_mask] = 255

    # Optional grow-from-strong-seed mode. This keeps large vessels from vanishing,
    # while avoiding some diffuse bright background in blurry images.
    if params.get("large_seed_strategy", "plain") == "strong_grow":
        strong_pct = float(params.get("large_seed_strong_percentile", 94))
        strong_threshold = np.percentile(roi_pixels, strong_pct)
        strong = np.zeros_like(source, dtype=np.uint8)
        strong[(source >= strong_threshold) & roi_mask] = 255

        grow_kernel = odd_kernel_size(
            image_size,
            float(params.get("large_seed_grow_kernel_scale", 0.012)),
            int(params.get("large_seed_grow_kernel_min", 7)),
        )
        grow_struct = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (grow_kernel, grow_kernel))
        grown = cv2.dilate(strong, grow_struct, iterations=int(params.get("large_seed_grow_iterations", 1)))
        candidate = cv2.bitwise_and(candidate, grown)

    close_kernel = odd_kernel_size(
        image_size,
        float(params.get("large_seed_close_kernel_scale", 0.008)),
        int(params.get("large_seed_close_kernel_min", 5)),
    )
    close_struct = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_kernel, close_kernel))
    large_seed = cv2.morphologyEx(
        candidate,
        cv2.MORPH_CLOSE,
        close_struct,
        iterations=int(params.get("large_seed_close_iterations", 1)),
    )
    return np.where(roi_mask, large_seed, 0).astype(np.uint8)


def run_onh_binarization(image: np.ndarray, roi_mask: np.ndarray, params: dict) -> dict:
    image = image.astype(np.float64)
    roi_mask = roi_mask.astype(bool)
    image_size = image.shape[0]
    working = np.where(roi_mask, image, 0).astype(np.float64)

    if params.get("median_filter", False):
        ksize = max(1, int(params.get("median_kernel_size", 3)))
        working = median_filter(working, size=(ksize, ksize), mode="constant", cval=0.0)
        working = np.where(roi_mask, working, 0).astype(np.float64)

    normalized = normalize_roi_to_uint8(
        working,
        roi_mask,
        low_pct=float(params.get("normalize_low_pct", 1)),
        high_pct=float(params.get("normalize_high_pct", 99)),
    )

    clahe = cv2.createCLAHE(
        clipLimit=float(params.get("clahe_clip", 2.0)),
        tileGridSize=tuple(params.get("clahe_grid", (8, 8))),
    )
    enhanced = clahe.apply(normalized)
    enhanced = np.where(roi_mask, enhanced, 0).astype(np.uint8)

    bg_kernel = odd_kernel_size(
        image_size,
        float(params.get("background_kernel_scale", 0.035)),
        int(params.get("background_kernel_min", 15)),
    )
    background = cv2.GaussianBlur(enhanced, (bg_kernel, bg_kernel), 0)
    corrected = cv2.addWeighted(
        enhanced,
        1.0,
        background,
        float(params.get("background_weight", -0.5)),
        0,
    )
    corrected = cv2.normalize(corrected, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    corrected = np.where(roi_mask, corrected, 0).astype(np.uint8)

    smooth_kernel = max(1, int(params.get("smooth_kernel", 3)))
    if smooth_kernel % 2 == 0:
        smooth_kernel += 1
    smoothed = cv2.GaussianBlur(corrected, (smooth_kernel, smooth_kernel), 0) if smooth_kernel > 1 else corrected

    large_seed = build_large_seed(enhanced, corrected, roi_mask, image_size, params)

    if params.get("use_frangi", True):
        frangi_min = int(params.get("frangi_min", 1))
        frangi_max = int(params.get("frangi_max", 4))
        if frangi_max < frangi_min:
            frangi_min, frangi_max = frangi_max, frangi_min
        frangi_opts = {
            "sigmarange": (frangi_min, frangi_max),
            "sigmastepsize": int(params.get("frangi_sigma_step", 1)),
            "correctionconst1": float(params.get("frangi_c1", 0.8)),
            "correctionconst2": float(params.get("frangi_c2", 15)),
        }
        frangi_raw = frangi_2Dfilter(smoothed.astype(np.float64), frangi_opts)
        frangi = cv2.normalize(frangi_raw, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        frangi = np.where(roi_mask, frangi, 0).astype(np.uint8)

        fine_threshold = np.percentile(frangi[roi_mask], float(params.get("fine_percentile", 74)))
        fine_candidate = np.zeros_like(frangi, dtype=np.uint8)
        fine_candidate[(frangi >= fine_threshold) & roi_mask] = 255

        # Optional weak intensity support. This can reduce isolated Frangi speckle,
        # but is deliberately mild so fine vessels are still visible.
        if params.get("fine_require_corrected_support", False):
            support_pct = float(params.get("fine_support_percentile", 35))
            support_threshold = np.percentile(corrected[roi_mask], support_pct)
            fine_candidate[(corrected < support_threshold) & roi_mask] = 0

        fine_kernel = odd_kernel_size(
            image_size,
            float(params.get("fine_kernel_scale", 0.004)),
            int(params.get("fine_kernel_min", 3)),
        )
        fine_struct = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (fine_kernel, fine_kernel))
        fine_candidate = cv2.morphologyEx(
            fine_candidate,
            cv2.MORPH_CLOSE,
            fine_struct,
            iterations=int(params.get("fine_close_iterations", 1)),
        )
        fine_candidate = np.where(roi_mask, fine_candidate, 0).astype(np.uint8)
    else:
        frangi = np.zeros_like(corrected, dtype=np.uint8)
        fine_candidate = np.zeros_like(corrected, dtype=np.uint8)

    binary = cv2.bitwise_or(large_seed, fine_candidate)
    binary = np.where(roi_mask, binary, 0).astype(np.uint8)

    cleanup_kernel = odd_kernel_size(
        image_size,
        float(params.get("cleanup_kernel_scale", 0.003)),
        int(params.get("cleanup_kernel_min", 3)),
    )
    cleanup_struct = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (cleanup_kernel, cleanup_kernel))
    binary = cv2.morphologyEx(
        binary,
        cv2.MORPH_CLOSE,
        cleanup_struct,
        iterations=int(params.get("cleanup_close_iterations", 1)),
    )
    binary = remove_small_components(binary, roi_mask, int(params.get("min_component_area_px", 0)))

    return {
        "masked_roi": normalize_for_display(np.where(roi_mask, image, 0), roi_mask),
        "normalized": normalized,
        "enhanced": enhanced,
        "corrected": corrected,
        "frangi": frangi,
        "large_seed": large_seed,
        "fine_candidate": fine_candidate,
        "binary": binary,
        "roi_mask": roi_mask,
        "params": params.copy(),
    }

## Baseline: Current-Like Large-Preserving Parameters

This baseline keeps the original ONH idea, but names the bright vessel mask `large_seed` to make the purpose explicit.

In [ ]:
BASELINE_PARAMS = {
    "median_filter": False,
    "median_kernel_size": 3,
    "normalize_low_pct": 1,
    "normalize_high_pct": 99,
    "clahe_clip": 2.0,
    "clahe_grid": (8, 8),
    "background_kernel_scale": 0.035,
    "background_kernel_min": 15,
    "background_weight": -0.5,
    "smooth_kernel": 3,
    "large_seed_source": "enhanced",
    "large_seed_strategy": "plain",
    "large_seed_percentile": 85,
    "large_seed_close_kernel_scale": 0.008,
    "large_seed_close_kernel_min": 5,
    "large_seed_close_iterations": 1,
    "use_frangi": True,
    "frangi_min": 1,
    "frangi_max": 4,
    "frangi_sigma_step": 1,
    "frangi_c1": 0.8,
    "frangi_c2": 15,
    "fine_percentile": 70,
    "fine_kernel_scale": 0.004,
    "fine_kernel_min": 3,
    "fine_close_iterations": 1,
    "fine_require_corrected_support": False,
    "cleanup_kernel_scale": 0.003,
    "cleanup_kernel_min": 3,
    "cleanup_close_iterations": 1,
    "min_component_area_px": 0,
}

baseline_results = {
    case["name"]: run_onh_binarization(case["image"], case["roi_mask"], BASELINE_PARAMS)
    for case in cases
}
print("Baseline complete.")

In [ ]:
def plot_stage_grid(results_by_name: dict, stages: list[str], figsize_per_cell: float = 2.35) -> None:
    names = list(results_by_name.keys())
    fig, axes = plt.subplots(
        len(names),
        len(stages),
        figsize=(figsize_per_cell * len(stages), figsize_per_cell * len(names)),
        squeeze=False,
    )
    for r, name in enumerate(names):
        result = results_by_name[name]
        for c, stage in enumerate(stages):
            ax = axes[r, c]
            ax.imshow(result[stage], cmap="gray", vmin=0, vmax=255)
            if r == 0:
                ax.set_title(stage)
            if c == 0:
                ax.set_ylabel(name, rotation=90, labelpad=8)
            ax.set_xticks([])
            ax.set_yticks([])
    plt.tight_layout()


plot_stage_grid(
    baseline_results,
    ["masked_roi", "enhanced", "corrected", "frangi", "large_seed", "fine_candidate", "binary"],
)

In [ ]:
def binary_overlay(image: np.ndarray, binary: np.ndarray, roi_mask: np.ndarray, alpha: float = 0.55) -> np.ndarray:
    base = normalize_for_display(image, roi_mask)
    rgb = np.dstack([base, base, base]).astype(np.float32)
    vessel = binary > 0
    overlay_color = np.array([255, 70, 70], dtype=np.float32)
    rgb[vessel] = (1 - alpha) * rgb[vessel] + alpha * overlay_color
    rgb[~roi_mask] = 0
    return np.clip(rgb, 0, 255).astype(np.uint8)


def plot_binary_overlay(cases: list[dict], results_by_name: dict, title: str = "Binary overlay") -> None:
    fig, axes = plt.subplots(1, len(cases), figsize=(4.2 * len(cases), 4.2))
    if len(cases) == 1:
        axes = [axes]
    for ax, case in zip(axes, cases):
        result = results_by_name[case["name"]]
        overlay = binary_overlay(case["image"], result["binary"], case["roi_mask"])
        ax.imshow(overlay)
        ax.set_title(case["name"])
        ax.axis("off")
    fig.suptitle(title, y=0.98)
    plt.tight_layout()


plot_binary_overlay(cases, baseline_results, title="Baseline binary overlay")

In [ ]:
def binary_qc_table(cases: list[dict], results_by_name: dict, method_name: str) -> pd.DataFrame:
    rows = []
    for case in cases:
        name = case["name"]
        result = results_by_name[name]
        binary = result["binary"] > 0
        roi_mask = case["roi_mask"]
        roi_area = int(roi_mask.sum())
        vessel_area = int((binary & roi_mask).sum())
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(result["binary"], connectivity=8)
        component_areas = stats[1:, cv2.CC_STAT_AREA] if num_labels > 1 else np.array([], dtype=int)
        rows.append(
            {
                "method": method_name,
                "case": name,
                "foreground_pct_in_roi": 100 * vessel_area / max(roi_area, 1),
                "component_count": int(max(num_labels - 1, 0)),
                "largest_component_area_px": int(component_areas.max()) if component_areas.size else 0,
                "median_component_area_px": float(np.median(component_areas)) if component_areas.size else 0.0,
            }
        )
    return pd.DataFrame(rows)


baseline_qc = binary_qc_table(cases, baseline_results, "baseline")
baseline_qc

## Three Reasonable Candidate Versions

All three versions should preserve large vessels. They differ in how much fine vessel texture/noise they keep.

- **Version 1 - balanced:** close to the current behavior, but slightly cleaner than baseline.
- **Version 2 - clean large-preserving:** stricter Frangi threshold and grow-from-strong large seed to reduce blurry bright leakage while keeping big vessels.
- **Version 3 - fine-sensitive large-preserving:** keeps more fine vessels than Version 2, but still uses an explicit large seed.

In [ ]:
CANDIDATE_PARAM_SETS = {
    "Version 1 - balanced": {
        "clahe_clip": 1.8,
        "background_weight": -0.6,
        "large_seed_source": "enhanced",
        "large_seed_strategy": "plain",
        "large_seed_percentile": 86,
        "large_seed_close_kernel_scale": 0.008,
        "frangi_max": 4,
        "fine_percentile": 72,
        "fine_kernel_scale": 0.004,
        "cleanup_kernel_scale": 0.003,
    },
    "Version 2 - clean large-preserving": {
        "clahe_clip": 1.5,
        "background_weight": -0.7,
        "large_seed_source": "enhanced",
        "large_seed_strategy": "strong_grow",
        "large_seed_percentile": 84,
        "large_seed_strong_percentile": 91,
        "large_seed_grow_kernel_scale": 0.016,
        "large_seed_grow_kernel_min": 11,
        "large_seed_close_kernel_scale": 0.009,
        "frangi_max": 3,
        "fine_percentile": 77,
        "fine_kernel_scale": 0.003,
        "fine_require_corrected_support": True,
        "fine_support_percentile": 28,
        "cleanup_kernel_scale": 0.002,
        "min_component_area_px": 3,
    },
    "Version 3 - fine-sensitive large-preserving": {
        "clahe_clip": 1.7,
        "background_weight": -0.65,
        "large_seed_source": "enhanced",
        "large_seed_strategy": "plain",
        "large_seed_percentile": 87,
        "large_seed_close_kernel_scale": 0.007,
        "frangi_max": 4,
        "fine_percentile": 69,
        "fine_kernel_scale": 0.003,
        "fine_require_corrected_support": True,
        "fine_support_percentile": 22,
        "cleanup_kernel_scale": 0.002,
    },
}


def make_candidate_params(overrides: dict) -> dict:
    params = BASELINE_PARAMS.copy()
    params.update(overrides)
    return params


candidate_results = {}
for method_name, overrides in CANDIDATE_PARAM_SETS.items():
    params = make_candidate_params(overrides)
    candidate_results[method_name] = {
        case["name"]: run_onh_binarization(case["image"], case["roi_mask"], params)
        for case in cases
    }

candidate_qc = pd.concat(
    [binary_qc_table(cases, results, method_name) for method_name, results in candidate_results.items()],
    ignore_index=True,
)
candidate_qc

In [ ]:
def plot_candidate_binary_grid(candidate_results: dict, cases: list[dict]) -> None:
    method_names = list(candidate_results.keys())
    fig, axes = plt.subplots(len(cases), len(method_names), figsize=(4.0 * len(method_names), 3.8 * len(cases)), squeeze=False)
    for r, case in enumerate(cases):
        for c, method_name in enumerate(method_names):
            ax = axes[r, c]
            result = candidate_results[method_name][case["name"]]
            ax.imshow(result["binary"], cmap="gray", vmin=0, vmax=255)
            if r == 0:
                ax.set_title(method_name)
            if c == 0:
                ax.set_ylabel(case["name"], rotation=90, labelpad=8)
            ax.set_xticks([])
            ax.set_yticks([])
    fig.suptitle("Final binary comparison: 4 test images x 3 reasonable candidate versions", y=0.995)
    plt.tight_layout()


plot_candidate_binary_grid(candidate_results, cases)

In [ ]:
def plot_candidate_overlay_grid(candidate_results: dict, cases: list[dict]) -> None:
    method_names = list(candidate_results.keys())
    fig, axes = plt.subplots(len(cases), len(method_names), figsize=(4.0 * len(method_names), 3.8 * len(cases)), squeeze=False)
    for r, case in enumerate(cases):
        for c, method_name in enumerate(method_names):
            ax = axes[r, c]
            result = candidate_results[method_name][case["name"]]
            overlay = binary_overlay(case["image"], result["binary"], case["roi_mask"])
            ax.imshow(overlay)
            if r == 0:
                ax.set_title(method_name)
            if c == 0:
                ax.set_ylabel(case["name"], rotation=90, labelpad=8)
            ax.set_xticks([])
            ax.set_yticks([])
    fig.suptitle("Overlay comparison: red = detected vessel pixels", y=0.995)
    plt.tight_layout()


plot_candidate_overlay_grid(candidate_results, cases)

In [ ]:
def plot_candidate_stage_grid(candidate_results: dict, cases: list[dict], stage: str) -> None:
    method_names = list(candidate_results.keys())
    fig, axes = plt.subplots(len(cases), len(method_names), figsize=(4.0 * len(method_names), 3.8 * len(cases)), squeeze=False)
    for r, case in enumerate(cases):
        for c, method_name in enumerate(method_names):
            ax = axes[r, c]
            result = candidate_results[method_name][case["name"]]
            ax.imshow(result[stage], cmap="gray", vmin=0, vmax=255)
            if r == 0:
                ax.set_title(method_name)
            if c == 0:
                ax.set_ylabel(case["name"], rotation=90, labelpad=8)
            ax.set_xticks([])
            ax.set_yticks([])
    fig.suptitle(f"Candidate {stage} comparison", y=0.995)
    plt.tight_layout()


plot_candidate_stage_grid(candidate_results, cases, "large_seed")
plot_candidate_stage_grid(candidate_results, cases, "fine_candidate")

## Optional: Inspect One Version In Detail

Change `METHOD_TO_INSPECT` to any candidate method name and rerun this cell if one version looks promising.

In [ ]:
METHOD_TO_INSPECT = "Version 2 - clean large-preserving"

plot_stage_grid(
    candidate_results[METHOD_TO_INSPECT],
    ["masked_roi", "enhanced", "corrected", "frangi", "large_seed", "fine_candidate", "binary"],
)
plot_binary_overlay(cases, candidate_results[METHOD_TO_INSPECT], title=f"{METHOD_TO_INSPECT} overlay")

## Optional: Save Figures

Set `SAVE_FIGURES = True` if you want PNGs saved to `algorithms test images/notebook_outputs`.

In [ ]:
SAVE_FIGURES = False

if SAVE_FIGURES:
    for method_name, results in candidate_results.items():
        safe_name = method_name.lower().replace(" ", "_").replace("-", "_")
        for case in cases:
            binary = results[case["name"]]["binary"]
            out_path = OUTPUT_DIR / f"{case['name'].replace(' ', '_')}_{safe_name}_binary.png"
            cv2.imwrite(str(out_path), binary)
    candidate_qc.to_csv(OUTPUT_DIR / "candidate_binary_qc.csv", index=False)
    print("Saved outputs to", OUTPUT_DIR)
else:
    print("SAVE_FIGURES is False; nothing was written.")

## How To Judge The Results

For this tuning pass, compare the final binary and overlay visually:

1. Large vessels should remain continuous and filled, not hollow, broken, or missing.
2. Fine radial capillaries should be present, but not converted into uniform noisy texture.
3. Bright blurry sectors should not become solid white regions.
4. The same parameter set should behave reasonably on A clear, A blurry, B clear, and B blurry before we move it into production code.